In [ ]:
!pip install deep-translator


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.9 MB/s eta 0:00:00


In [ ]:
import gdown

file_id = '1iCjRmhDNrhyrXzsIeWzOpGSBo2K3vhYM' # Extracted from the Google Drive link
output_file = 'dataset_requisitos_ingles.csv'
gdown.download(f'https://drive.google.com/uc?id={file_id}', output_file, fuzzy=True)
print(f'Downloaded {output_file} successfully.')

Downloading...
From: https://drive.google.com/uc?id=1iCjRmhDNrhyrXzsIeWzOpGSBo2K3vhYM
To: /content/dataset_requisitos_ingles.csv
100%|██████████| 1.95M/1.95M [00:00<00:00, 160MB/s]

Downloaded dataset_requisitos_ingles.csv successfully.


In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator, exceptions
import time

def translate_sentence_column(df: pd.DataFrame, source_column: str = 'sentence', target_column: str = 'sentence_espanol', max_retries: int = 5, delay_seconds: int = 1) -> pd.DataFrame:
    print(f"Iniciando traducción de la columna '{source_column}' a '{target_column}'... Esto puede tardar varios minutos dependiendo del tamaño del dataset.\n           Nota: El traductor gratuito puede tener limitaciones de uso. Las oraciones no traducidas se mantendrán en inglés.")
    translator = GoogleTranslator(source='en', target='es')

    def traducir_texto(texto):
        if pd.isna(texto) or not str(texto).strip():
            return ''
        text_to_translate = str(texto)
        for attempt in range(max_retries):
            try:
                translated_text = translator.translate(text_to_translate)
                if translated_text == text_to_translate and text_to_translate != '': # If translation returned original text, it likely failed or hit a limit
                    raise exceptions.TranslationNotFound(f"No translation found for: '{text_to_translate}'")
                return translated_text
            except (exceptions.TranslationNotFound, exceptions.TooManyRequests) as e:
                print(f"Advertencia al traducir '{text_to_translate}' (Intento {attempt + 1}/{max_retries}): {e}. Reintentando...")
                time.sleep(delay_seconds * (attempt + 1))
            except Exception as e:
                print(f"Error inesperado al traducir '{text_to_translate}' (Intento {attempt + 1}/{max_retries}): {e}. Reintentando...")
                time.sleep(delay_seconds * (attempt + 1))
        print(f"Fallo la traducción después de {max_retries} intentos para: '{text_to_translate}'. Se mantendrá el texto original.")
        return text_to_translate

    df[target_column] = df[source_column].apply(traducir_texto)
    print(f"Traducción de la columna '{source_column}' finalizada.")
    return df


In [ ]:
def classify_requirements_in_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clasifica los requisitos de un DataFrame como Funcionales (RF) o No Funcionales (RNF)
    basándose en las columnas 'NFR_boolean', 'security' y 'reliability'.
    """
    print("Iniciando clasificación de requisitos (RF / RNF)...")
    def clasificar_row(row):
        if row.get('NFR_boolean', 0) == 1 or row.get('security', 0) == 1 or row.get('reliability', 0) == 1:
            return 'RNF'
        return 'RF'

    df['tipo_requisito'] = df.apply(clasificar_row, axis=1)
    print("Clasificación de requisitos finalizada.")
    return df

In [ ]:
# --- Ejecución del proceso completo ---

# Rutas de los archivos de entrada y salida
INPUT_CSV = "dataset_requisitos_ingles.csv"
OUTPUT_CSV = "dataset_requisitos_traducido_clasificado.csv"

print("Proceso iniciado: Clasificación y Traducción de Requisitos.")

# 1. Cargar el dataset
print("Cargando dataset desde: " + INPUT_CSV)
df_raw = pd.read_csv(INPUT_CSV, encoding='latin1')

# 2. Clasificar los requisitos primero
df_classified = classify_requirements_in_df(df_raw.copy()) # Usar una copia
df_classified
# 3. Traducir la columna 'sentence'
df_processed = translate_sentence_column(df_classified.copy()) # Usar una copia

# 4. Estructurar el DataFrame resultante para guardar
df_resultado_final = df_processed[['id', 'sentence', 'sentence_espanol', 'tipo_requisito', 'security', 'reliability', 'NFR_boolean']]

# 5. Guardar archivo procesado
df_resultado_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"Proceso finalizado exitosamente. Archivo guardado en: {OUTPUT_CSV}")

# Mostrar vista previa en consola
#print("\n--- VISTA PREVIA DEL RESULTADO ---")
#display(df_resultado_final[['sentence_espanol', 'tipo_requisito']].head(10))

#print(f"\nPara descargar el archivo '{OUTPUT_CSV}', haz clic en el icono de la carpeta (Explorador de archivos) en el panel lateral izquierdo de Colab. Luego, navega hasta la ubicación del archivo y haz clic derecho sobre él para seleccionarr 'Descargar'.")

Proceso iniciado: Clasificación y Traducción de Requisitos.
Cargando dataset desde: dataset_requisitos_ingles.csv
Iniciando clasificación de requisitos (RF / RNF)...
Clasificación de requisitos finalizada.
Iniciando traducción de la columna 'sentence' a 'sentence_espanol'... Esto puede tardar varios minutos dependiendo del tamaño del dataset.
           Nota: El traductor gratuito puede tener limitaciones de uso. Las oraciones no traducidas se mantendrán en inglés.
Advertencia al traducir 'The system shall create a single patient record for each patient.' (Intento 1/5): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function. Reintentando...
Advertencia al traducir 'The system shall create a single patient record for each patient.' (Intento 2/5): Server Error: You made too many requests to the server.According to g